In [1]:
# Install core dependencies for NF4/INT8/BF16
!pip install -U transformers accelerate bitsandbytes>=0.46.1 datasets

print("Install complete! If Colab asks you to restart the session again")

Install complete! If Colab asks you to restart the session again


In [2]:
import torch
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Verify bitsandbytes loaded correctly
print(f"BitsAndBytes version: {bnb.__version__}")

# Configuration
QUANT_MODE = "nf4"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

def get_quant_config(mode):
    if mode == "int8":
        return BitsAndBytesConfig(load_in_8bit=True)
    elif mode == "nf4":
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16
        )
    return None

# 1. Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# 2. Load Model
print(f"Loading {MODEL_ID} in {QUANT_MODE} mode... (This may take a minute)")
q_config = get_quant_config(QUANT_MODE)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=q_config,
    torch_dtype=torch.bfloat16 if QUANT_MODE == "bf16" else None,
    device_map="auto",
    low_cpu_mem_usage=True
)

# 3. Memory Check
print(f"Success! Model loaded.")
print(f"Peak GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

BitsAndBytes version: 0.49.2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Success! Model loaded.
Peak GPU Memory Allocated: 3951.07 MB


In [3]:
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [4]:
from datasets import load_dataset
import numpy as np

# 1. Load the dataset into the variable 'dataset'
dataset = load_dataset("openai/gsm8k", "main")

print("📊 GSM8K Dataset Statistics")
print("="*30)

# 2. Iterate through 'dataset'
for split_name in dataset.keys():
    split_data = dataset[split_name]
    print(f"\n--- {split_name.upper()} SPLIT ---")
    print(f"Total Problems: {split_data.num_rows}")

    # Calculate character lengths for questions and answers
    q_lengths = [len(row['question']) for row in split_data]
    a_lengths = [len(row['answer']) for row in split_data]

    # Print the stats
    print(f"Average Question Length: {np.mean(q_lengths):.1f} characters")
    print(f"Longest Question:        {np.max(q_lengths)} characters")
    print(f"Average Answer Length:   {np.mean(a_lengths):.1f} characters")
    print(f"Longest Answer:          {np.max(a_lengths)} characters")

📊 GSM8K Dataset Statistics

--- TRAIN SPLIT ---
Total Problems: 7473
Average Question Length: 234.5 characters
Longest Question:        985 characters
Average Answer Length:   287.5 characters
Longest Answer:          1228 characters

--- TEST SPLIT ---
Total Problems: 1319
Average Question Length: 239.9 characters
Longest Question:        848 characters
Average Answer Length:   292.9 characters
Longest Answer:          1070 characters


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!pip install gptqmodel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.6/885.6 kB 16.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pypcre-0.3.2-cp312-cp312-linux_x86_64.whl
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 12.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 8.2 MB/s eta 0:00:00
  Installing 

In [1]:
###############################################################################
# GPTQ LOAD TEST v2 — using GPTQModel (successor to dead auto-gptq)
# + RedHatAI's v0.3 GPTQ checkpoint (same model version = fair comparison)
# Paste as a single Colab cell. GPU runtime required.
###############################################################################

# ── 1. Install GPTQModel + deps ─────────────────────────────────────────────
# GPTQModel needs torch visible at build time → --no-build-isolation
# If this fails, try: pip install gptqmodel --no-build-isolation
import subprocess, sys

print("Installing dependencies (this may take 2-3 min for CUDA kernels)...")

for cmd in [
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "accelerate", "optimum", "transformers"],
    [sys.executable, "-m", "pip", "install", "-q",
     "gptqmodel", "--no-build-isolation"],
]:
    print(f"  Running: {' '.join(cmd[-3:])}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ⚠️  FAILED — stderr (last 500 chars):")
        print(result.stderr[-500:])
        print("\n  Trying alternative install from source...")
        # Fallback: install from GitHub
        alt = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "git+https://github.com/ModelCloud/GPTQModel.git",
             "--no-build-isolation"],
            capture_output=True, text=True,
        )
        if alt.returncode != 0:
            print(f"  ❌ Source install also failed.")
            print(alt.stderr[-500:])
            raise SystemExit("Cannot install GPTQModel. See errors above.")
        print("  ✔ Installed from source")

print("✔ All packages installed")

# ── 2. Environment check ────────────────────────────────────────────────────
import torch, importlib.metadata

print(f"\n{'='*50}")
print("ENVIRONMENT")
print(f"{'='*50}")
print(f"  torch:          {torch.__version__}")
print(f"  CUDA:           {torch.version.cuda}")
print(f"  GPU:            {torch.cuda.get_device_name(0)}")
print(f"  VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

for lib in ["transformers", "optimum", "gptqmodel"]:
    try:
        v = importlib.metadata.version(lib)
        print(f"  {lib:<16} {v}")
    except:
        print(f"  {lib:<16} NOT FOUND ❌")

# ── 3. Load the v0.3 GPTQ model ─────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM

# This is a GPTQ-4bit of the SAME v0.3 you use for BF16/INT8/NF4
# No version mismatch!
MODEL_ID = "RedHatAI/Mistral-7B-Instruct-v0.3-GPTQ-4bit"

print(f"\n{'='*50}")
print(f"Testing: {MODEL_ID}")
print(f"{'='*50}")

try:
    print("  Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("  ✔ Tokenizer loaded")

    print("  Loading model (~4GB download)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    print("  ✔ Model loaded")

    mem = torch.cuda.max_memory_allocated() / 1e6
    print(f"  Peak VRAM: {mem:.0f} MB")

except Exception as e:
    print(f"\n  ❌ LOAD FAILED: {e}")
    print("\n  If GPTQModel CUDA kernels didn't compile, try:")
    print("    1. Runtime → Restart runtime")
    print("    2. Then re-run this cell")
    print("  (CUDA kernels sometimes need a fresh runtime after install)")
    raise SystemExit

# ── 4. Single inference test ─────────────────────────────────────────────────
import time

question = ("Natalia sold clips to 48 of her friends in April, "
            "and then she sold half as many clips in May. "
            "How many clips did Natalia sell altogether in April and May?")

messages = [{"role": "user",
             "content": "Show your reasoning step by step.\n\n" + question}]

try:
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
except:
    prompt = f"[INST] Show your reasoning step by step.\n\n{question} [/INST]"

inputs   = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
input_ids = inputs["input_ids"].to(model.device)
attn_mask = inputs["attention_mask"].to(model.device)
input_len = input_ids.shape[1]

torch.cuda.synchronize()
t0 = time.perf_counter()

with torch.inference_mode():
    outputs = model.generate(
        input_ids,
        attention_mask=attn_mask,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

torch.cuda.synchronize()
t1 = time.perf_counter()

gen_ids = outputs[0, input_len:]
answer  = tokenizer.decode(gen_ids, skip_special_tokens=True)

import re
nums = re.findall(r"-?\d[\d,]*\.?\d*", answer)
predicted = nums[-1].replace(",", "") if nums else "?"

print(f"\n{'='*50}")
print("INFERENCE TEST")
print(f"{'='*50}")
print(f"  Tokens generated: {len(gen_ids)}")
print(f"  Latency:          {t1 - t0:.2f}s")
print(f"  Throughput:       {len(gen_ids) / (t1 - t0):.1f} tok/s")
print(f"  Extracted answer: {predicted}")
print(f"  Expected:         72")
print(f"  Correct:          {'✔ YES' if predicted == '72' else '✗ NO'}")
print(f"\n  Output (last 300 chars):")
print(f"  ...{answer[-300:]}")

# ── 5. Cleanup ──────────────────────────────────────────────────────────────
import gc
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

print(f"\n{'='*50}")
print("✅ GPTQ TEST PASSED")
print(f"{'='*50}")
print(f"  Model: {MODEL_ID}")
print(f"  This is Mistral v0.3 — same version as your BF16/INT8/NF4")
print(f"  → Fair apples-to-apples comparison ✔")
print(f"\n  To add GPTQ to your main experiment, update CONFIGS:")
print(f'  {{"name": "GPTQ", "model_id": "{MODEL_ID}", "quant": "gptq"}}')

Installing dependencies (this may take 2-3 min for CUDA kernels)...
  Running: accelerate optimum transformers
  Running: -q gptqmodel --no-build-isolation
✔ All packages installed

ENVIRONMENT
  torch:          2.10.0+cu128
  CUDA:           12.8
  GPU:            NVIDIA L4
  VRAM:           23.7 GB
  transformers     5.6.2
  optimum          2.1.0
  gptqmodel        6.0.3

Testing: RedHatAI/Mistral-7B-Instruct-v0.3-GPTQ-4bit
  Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/972 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

INFO  Format: Converting `format` from `FORMAT.GPTQ` to internal `FORMAT.GPTQ_V2`.


INFO  Format: Converting GPTQ v1 to v2                                         


INFO  Optimize: `TritonV2QuantLinear` compilation triggered.                   


INFO  gc.collect() reclaimed 95 objects in 0.279s                              


  ✔ Model loaded
  Peak VRAM: 4168 MB

INFERENCE TEST
  Tokens generated: 226
  Latency:          31.23s
  Throughput:       7.2 tok/s
  Extracted answer: 48
  Expected:         72
  Correct:          ✗ NO

  Output (last 300 chars):
  ...n April and May, we would need to know the number of clips she sold per friend and then multiply that by the number of friends (48) for April, and then by half of that number for May.

Since we don't have enough information, we can't determine the total number of clips Natalia sold in April and May.

✅ GPTQ TEST PASSED
  Model: RedHatAI/Mistral-7B-Instruct-v0.3-GPTQ-4bit
  This is Mistral v0.3 — same version as your BF16/INT8/NF4
  → Fair apples-to-apples comparison ✔

  To add GPTQ to your main experiment, update CONFIGS:
  {"name": "GPTQ", "model_id": "RedHatAI/Mistral-7B-Instruct-v0.3-GPTQ-4bit", "quant": "gptq"}


In [2]:
###############################################################################
# MISTRAL 7B — GSM8K QUANTIZATION ANALYSIS (BF16, INT8, NF4, GPTQ)
# Single cell — copy-paste into Colab. GPU runtime required (L4 recommended).
# Drive must be mounted at /content/drive
# ── FAST VERSION: 150 problems, tqdm, speed optimizations ──
###############################################################################

# ── 0. Installs ──────────────────────────────────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    for p in pkgs:
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", p],
                stderr=subprocess.DEVNULL,
            )
        except subprocess.CalledProcessError:
            print(f"⚠️  Could not install {p} — skipping")

def pip_install_special(pkg, *extra_args):
    """For packages needing special flags like --no-build-isolation."""
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pkg, *extra_args],
            stderr=subprocess.DEVNULL,
        )
    except subprocess.CalledProcessError:
        print(f"⚠️  Could not install {pkg} — skipping")

pip_install(
    "transformers>=4.40",
    "accelerate>=0.30",
    "bitsandbytes>=0.43",
    "optimum>=1.19",
    "datasets",
    "sentencepiece",
    "protobuf",
    "tqdm",
)

# GPTQModel needs torch visible at build time
pip_install_special("gptqmodel", "--no-build-isolation")

# ── 1. Imports ───────────────────────────────────────────────────────────────
import os, re, gc, json, time, random
import numpy as np
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# ── 2. Paths & constants ────────────────────────────────────────────────────
SEED = 42
SAVE_DIR = "/content/drive/MyDrive/Papers/Wu"
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_CACHE = "/content/model_cache"
os.makedirs(MODEL_CACHE, exist_ok=True)

DEVICE = "cuda"
assert torch.cuda.is_available(), "GPU required — switch runtime to GPU."

# ── SPEED KNOBS ──────────────────────────────────────────────────────────────
MAX_NEW_TOKENS     = 256        # most GSM8K answers finish <200 tok
SAMPLES_PER_BUCKET = 50         # 150 total (50 easy + 50 medium + 50 hard)
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = "You are a helpful math assistant. Show your reasoning step by step."

BASE_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
GPTQ_MODEL_ID = "RedHatAI/Mistral-7B-Instruct-v0.3-GPTQ-4bit"

CONFIGS = [
    {"name": "BF16", "model_id": BASE_MODEL_ID, "quant": None},
    {"name": "INT8", "model_id": BASE_MODEL_ID, "quant": "int8"},
    {"name": "NF4",  "model_id": BASE_MODEL_ID, "quant": "nf4"},
    {"name": "GPTQ", "model_id": GPTQ_MODEL_ID, "quant": "gptq"},
]

# ── 3. Smart caching — reuse indices & skip finished configs ────────────────
INDICES_PATH  = os.path.join(SAVE_DIR, "eval_indices.json")
COMBINED_PATH = os.path.join(SAVE_DIR, "mistral_all_results.json")

def load_existing_results():
    """Load any per-config results already on Drive."""
    done = {}
    for cfg in CONFIGS:
        p = os.path.join(SAVE_DIR, f"mistral_{cfg['name'].lower()}_results.json")
        if os.path.exists(p):
            try:
                with open(p) as f:
                    data = json.load(f)
                if data.get("summary", {}).get("status") == "COMPLETED":
                    done[cfg["name"]] = data
                    print(f"  ✔ Found completed results for {cfg['name']} — will skip")
            except Exception:
                pass
    return done

print("=" * 60)
print("CHECKING FOR CACHED RESULTS ON DRIVE")
print("=" * 60)
existing = load_existing_results()
if not existing:
    print("  No cached results found — will run all configs.")

# ── 4. Stratified sampling (load from cache or create) ──────────────────────
print("\n" + "=" * 60)
print("STEP 1: Stratified sampling")
print("=" * 60)

def count_steps(solution: str) -> int:
    sentences = re.split(r'(?<=[.!?])\s+', solution.strip())
    return max(1, len([s for s in sentences if len(s.strip()) > 5]))

def assign_bucket(n: int) -> str:
    if n <= 3:  return "easy"
    if n <= 5:  return "medium"
    return "hard"

if os.path.exists(INDICES_PATH):
    print(f"  Loading cached indices from {INDICES_PATH}")
    with open(INDICES_PATH) as f:
        idx_data = json.load(f)
    sampled_indices = idx_data["indices"]
    bucket_labels   = {int(k): v for k, v in idx_data["bucket_labels"].items()}
    step_counts     = {int(k): v for k, v in idx_data["step_counts"].items()}
    print(f"  Loaded {len(sampled_indices)} indices "
          f"(Easy: {sum(1 for v in bucket_labels.values() if v=='easy')}, "
          f"Medium: {sum(1 for v in bucket_labels.values() if v=='medium')}, "
          f"Hard: {sum(1 for v in bucket_labels.values() if v=='hard')})")
else:
    print("  No cached indices — creating stratified sample...")
    dataset_full = load_dataset("openai/gsm8k", "main", split="test")
    print(f"  Full test set: {len(dataset_full)} problems")

    bucketed = {"easy": [], "medium": [], "hard": []}
    step_counts = {}
    for idx in range(len(dataset_full)):
        n = count_steps(dataset_full[idx]["answer"])
        step_counts[idx] = n
        bucketed[assign_bucket(n)].append(idx)

    print(f"  Pool — Easy: {len(bucketed['easy'])}, "
          f"Medium: {len(bucketed['medium'])}, Hard: {len(bucketed['hard'])}")

    random.seed(SEED)
    sampled_indices = []
    bucket_labels = {}
    for b in ["easy", "medium", "hard"]:
        chosen = random.sample(bucketed[b], min(SAMPLES_PER_BUCKET, len(bucketed[b])))
        for idx in chosen:
            bucket_labels[idx] = b
        sampled_indices.extend(chosen)

    with open(INDICES_PATH, "w") as f:
        json.dump({
            "seed": SEED,
            "samples_per_bucket": SAMPLES_PER_BUCKET,
            "indices": sampled_indices,
            "bucket_labels": {str(k): v for k, v in bucket_labels.items()},
            "step_counts":   {str(k): v for k, v in step_counts.items()},
        }, f, indent=2)
    print(f"  Saved {len(sampled_indices)} indices → {INDICES_PATH}")

# Load the actual dataset rows
dataset = load_dataset("openai/gsm8k", "main", split="test")
eval_subset = dataset.select(sampled_indices)
print(f"  Eval subset ready: {len(eval_subset)} problems")

# ── 5. Ensure models are downloaded (cache for reuse) ───────────────────────
print("\n" + "=" * 60)
print("STEP 2: Checking model availability")
print("=" * 60)

print(f"  Ensuring {BASE_MODEL_ID} weights are cached...")
_ = AutoTokenizer.from_pretrained(BASE_MODEL_ID, cache_dir=MODEL_CACHE)
print("  ✔ Tokenizer cached")

needs_base = any(
    cfg["model_id"] == BASE_MODEL_ID and cfg["name"] not in existing
    for cfg in CONFIGS
)
if needs_base:
    print(f"  Pre-downloading base model weights (this only happens once)...")
    _m = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        dtype=torch.bfloat16,
        cache_dir=MODEL_CACHE,
        device_map="cpu",
    )
    del _m; gc.collect()
    print("  ✔ Base model weights cached")
else:
    print("  ✔ Base model not needed (all base configs done)")

needs_gptq = "GPTQ" not in existing
if needs_gptq:
    print(f"  Ensuring {GPTQ_MODEL_ID} is cached...")
    try:
        _ = AutoTokenizer.from_pretrained(GPTQ_MODEL_ID, cache_dir=MODEL_CACHE)
        print("  ✔ GPTQ tokenizer cached")
    except Exception as e:
        print(f"  ⚠️  GPTQ tokenizer issue: {e}")
        print("     Will attempt at runtime")

# ── 6. Answer extraction ────────────────────────────────────────────────────

def normalize_number(x):
    if x is None: return None
    x = str(x).strip().replace(",", "")
    return x if x != "" else None

def extract_answer(text):
    if not text: return None
    m = re.search(r"####\s*(-?[0-9\.,]+)", text)
    if m: return normalize_number(m.group(1))
    matches = re.findall(r"-?\d[\d,]*\.?\d*", text)
    return normalize_number(matches[-1]) if matches else None

def extract_gold_answer(answer_text):
    m = re.search(r"####\s*(-?[0-9\.,]+)", answer_text)
    if m: return normalize_number(m.group(1))
    return None

def numerically_equal(a, b, tol=1e-6):
    if a is None or b is None: return False
    try: return abs(float(a) - float(b)) <= tol
    except (ValueError, TypeError): return str(a).strip() == str(b).strip()

# ── 7. Model loading ────────────────────────────────────────────────────────

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    time.sleep(2)

def load_model_and_tokenizer(config):
    model_id = config["model_id"]
    quant    = config["quant"]

    tokenizer = AutoTokenizer.from_pretrained(
        model_id, cache_dir=MODEL_CACHE, trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    load_kwargs = dict(
        pretrained_model_name_or_path=model_id,
        cache_dir=MODEL_CACHE,
        device_map="auto",
        trust_remote_code=True,
    )

    if quant is None:                       # BF16
        load_kwargs["dtype"] = torch.bfloat16

    elif quant == "int8":
        load_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)

    elif quant == "nf4":
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

    elif quant == "gptq":
        pass  # pre-quantized, loads via GPTQModel backend automatically

    model = AutoModelForCausalLM.from_pretrained(**load_kwargs)
    model.eval()
    return model, tokenizer

# ── 8. Inference helper ─────────────────────────────────────────────────────

@torch.inference_mode()
def run_inference(model, tokenizer, question_text):
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + question_text},
    ]
    try:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        prompt = f"[INST] {SYSTEM_PROMPT}\n\n{question_text} [/INST]"

    inputs    = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    input_ids = inputs["input_ids"].to(model.device)
    attn_mask = inputs["attention_mask"].to(model.device)
    input_len = input_ids.shape[1]

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    outputs = model.generate(
        input_ids,
        attention_mask=attn_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

    torch.cuda.synchronize()
    t1 = time.perf_counter()

    generated_ids = outputs[0, input_len:]
    output_text   = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return output_text, len(generated_ids), t1 - t0

# ── 9. Main experiment loop ─────────────────────────────────────────────────
all_per_question = []
all_summaries    = []

for cfg_name, data in existing.items():
    all_summaries.append(data["summary"])
    all_per_question.extend(data["per_question"])

configs_to_run = [c for c in CONFIGS if c["name"] not in existing]

if not configs_to_run:
    print("\n✅ All configs already completed! Loading cached results.")
else:
    print(f"\n{'='*60}")
    print(f"STEP 3: Running {len(configs_to_run)} configs "
          f"({', '.join(c['name'] for c in configs_to_run)})")
    print(f"{'='*60}")

for cfg_idx, config in enumerate(configs_to_run):
    print(f"\n{'='*60}")
    print(f"CONFIG {cfg_idx+1}/{len(configs_to_run)}: Mistral-7B — {config['name']}")
    print(f"{'='*60}")

    free_gpu()

    try:
        print(f"  Loading model...")
        model, tokenizer = load_model_and_tokenizer(config)
        print(f"  ✔ Model loaded")
    except Exception as e:
        print(f"  ⚠️  FAILED to load {config['name']}: {e}")
        all_summaries.append({
            "model": "Mistral-7B-Instruct-v0.3",
            "quantization": config["name"],
            "status": "LOAD_FAILED",
            "error": str(e),
        })
        continue

    # Warmup (2 passes to let CUDA kernels compile)
    print("  Warmup pass...")
    try:
        _ = run_inference(model, tokenizer, "What is 2 + 2?")
        _ = run_inference(model, tokenizer, "What is 10 * 5?")
    except Exception as e:
        print(f"  Warmup issue (continuing): {e}")

    torch.cuda.reset_peak_memory_stats()

    correct_count = 0
    total_tokens  = 0
    total_latency = 0.0
    per_question_results = []

    n = len(eval_subset)
    pbar = tqdm(range(n), desc=f"  {config['name']}", unit="q",
                bar_format="  {l_bar}{bar:30}{r_bar}")

    for i in pbar:
        row          = eval_subset[i]
        original_idx = sampled_indices[i]
        gold_answer  = extract_gold_answer(row["answer"])
        bucket       = bucket_labels[original_idx]
        steps        = step_counts[original_idx]

        try:
            output_text, tokens_gen, latency = run_inference(
                model, tokenizer, row["question"]
            )
            predicted  = extract_answer(output_text)
            is_correct = numerically_equal(predicted, gold_answer)
        except Exception as e:
            output_text = f"ERROR: {e}"
            tokens_gen, latency, predicted, is_correct = 0, 0.0, None, False

        if is_correct: correct_count += 1
        total_tokens  += tokens_gen
        total_latency += latency

        record = {
            "model": "Mistral-7B-Instruct-v0.3",
            "quantization": config["name"],
            "prompt_type": "cot",
            "decoding": "greedy",
            "question_id": original_idx,
            "question_text": row["question"],
            "gold_answer": gold_answer,
            "prediction": predicted,
            "correct": is_correct,
            "tokens_generated": tokens_gen,
            "latency_sec": round(latency, 4),
            "output_text": output_text,
            "difficulty_bucket": bucket,
            "reasoning_steps": steps,
        }
        per_question_results.append(record)
        all_per_question.append(record)

        acc_so_far = correct_count / (i + 1) * 100
        pbar.set_postfix(acc=f"{acc_so_far:.1f}%", lat=f"{latency:.1f}s")

    pbar.close()

    # ── Summary ──────────────────────────────────────────────────────────
    peak_mem = torch.cuda.max_memory_allocated() / 1e6

    accuracy    = correct_count / n
    avg_latency = total_latency / n
    throughput  = total_tokens / total_latency if total_latency > 0 else 0

    bucket_acc = {}
    for b in ["easy", "medium", "hard"]:
        br = [r for r in per_question_results if r["difficulty_bucket"] == b]
        bc = sum(1 for r in br if r["correct"])
        bucket_acc[b] = {
            "correct": bc, "total": len(br),
            "accuracy": bc / len(br) if br else 0,
        }

    summary = {
        "model": "Mistral-7B-Instruct-v0.3",
        "quantization": config["name"],
        "model_id": config["model_id"],
        "prompt_type": "cot",
        "decoding": "greedy",
        "status": "COMPLETED",
        "accuracy": round(accuracy, 4),
        "accuracy_pct": round(accuracy * 100, 2),
        "correct": correct_count,
        "total": n,
        "avg_latency_sec": round(avg_latency, 4),
        "throughput_tokens_per_sec": round(throughput, 2),
        "avg_tokens_generated": round(total_tokens / n, 1),
        "peak_memory_mb": round(peak_mem, 1),
        "total_inference_time_min": round(total_latency / 60, 2),
        "bucket_accuracy": bucket_acc,
    }
    all_summaries.append(summary)

    print(f"\n  ── {config['name']} RESULTS ──")
    print(f"  Accuracy:     {accuracy*100:.1f}% ({correct_count}/{n})")
    print(f"  Easy:         {bucket_acc['easy']['accuracy']*100:.1f}%")
    print(f"  Medium:       {bucket_acc['medium']['accuracy']*100:.1f}%")
    print(f"  Hard:         {bucket_acc['hard']['accuracy']*100:.1f}%")
    print(f"  Avg latency:  {avg_latency:.2f}s")
    print(f"  Throughput:   {throughput:.1f} tok/s")
    print(f"  Peak memory:  {peak_mem:.0f} MB")

    cfg_path = os.path.join(SAVE_DIR, f"mistral_{config['name'].lower()}_results.json")
    with open(cfg_path, "w") as f:
        json.dump({"summary": summary, "per_question": per_question_results},
                  f, indent=2, default=str)
    print(f"  Saved → {cfg_path}")

    del model, tokenizer
    free_gpu()
    print(f"  ✔ Model unloaded")

# ── 10. Save combined results ───────────────────────────────────────────────
print(f"\n{'='*60}")
print("SAVING COMBINED RESULTS")
print(f"{'='*60}")

combined = {
    "experiment": "Mistral-7B-Instruct-v0.3 Quantization Analysis",
    "dataset": f"GSM8K (stratified {len(sampled_indices)}-problem sample)",
    "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS,
    "system_prompt": SYSTEM_PROMPT,
    "num_problems": len(sampled_indices),
    "sampling": {
        "method": "stratified",
        "per_bucket": SAMPLES_PER_BUCKET,
        "buckets": {
            "easy":   "2-3 reasoning steps",
            "medium": "4-5 reasoning steps",
            "hard":   "6+ reasoning steps",
        },
    },
    "configs_attempted": [c["name"] for c in CONFIGS],
    "summaries": all_summaries,
    "per_question_results": all_per_question,
}

with open(COMBINED_PATH, "w") as f:
    json.dump(combined, f, indent=2, default=str)
print(f"  Combined   → {COMBINED_PATH}")

summary_path = os.path.join(SAVE_DIR, "mistral_summary.json")
with open(summary_path, "w") as f:
    json.dump(all_summaries, f, indent=2)
print(f"  Summary    → {summary_path}")

# Environment info
import transformers, bitsandbytes
env_info = {
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
}
try:
    import importlib.metadata
    env_info["gptqmodel"] = importlib.metadata.version("gptqmodel")
except: pass

env_path = os.path.join(SAVE_DIR, "mistral_environment.json")
with open(env_path, "w") as f:
    json.dump(env_info, f, indent=2)
print(f"  Env info   → {env_path}")

# ── 11. Final table ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("FINAL RESULTS — MISTRAL 7B INSTRUCT v0.3")
print(f"{'='*60}")
print(f"{'Config':<8} {'Acc%':>6} {'Easy%':>6} {'Med%':>6} {'Hard%':>6} "
      f"{'Δ(H-E)':>7} {'Lat(s)':>7} {'Tok/s':>7} {'Mem(MB)':>8}")
print("-" * 72)

for s in all_summaries:
    if s.get("status") != "COMPLETED":
        print(f"{s['quantization']:<8} {'FAILED':>6}  — {s.get('error','')[:40]}")
        continue
    ba = s["bucket_accuracy"]
    delta = ba["hard"]["accuracy"] - ba["easy"]["accuracy"]
    print(f"{s['quantization']:<8} "
          f"{s['accuracy_pct']:>5.1f}% "
          f"{ba['easy']['accuracy']*100:>5.1f}% "
          f"{ba['medium']['accuracy']*100:>5.1f}% "
          f"{ba['hard']['accuracy']*100:>5.1f}% "
          f"{delta*100:>+6.1f}% "
          f"{s['avg_latency_sec']:>7.2f} "
          f"{s['throughput_tokens_per_sec']:>7.1f} "
          f"{s['peak_memory_mb']:>8.0f}")

print(f"\n✅ Done! All files in: {SAVE_DIR}")

CHECKING FOR CACHED RESULTS ON DRIVE
  No cached results found — will run all configs.

STEP 1: Stratified sampling
  Loading cached indices from /content/drive/MyDrive/Papers/Wu/eval_indices.json
  Loaded 150 indices (Easy: 50, Medium: 50, Hard: 50)
  Eval subset ready: 150 problems

STEP 2: Checking model availability
  Ensuring mistralai/Mistral-7B-Instruct-v0.3 weights are cached...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

INFO  Format: Converting `format` from `FORMAT.GPTQ` to internal `FORMAT.GPTQ_V2`.


INFO  gc.collect() reclaimed 237 objects in 0.285s                             


  ✔ Model loaded
  Warmup pass...


    GPTQ:   0%|                              | 0/150 [00:00<?, ?q/s]


  ── GPTQ RESULTS ──
  Accuracy:     34.0% (51/150)
  Easy:         40.0%
  Medium:       42.0%
  Hard:         20.0%
  Avg latency:  27.51s
  Throughput:   8.2 tok/s
  Peak memory:  8521 MB
  Saved → /content/drive/MyDrive/Papers/Wu/mistral_gptq_results.json
  ✔ Model unloaded

SAVING COMBINED RESULTS
  Combined   → /content/drive/MyDrive/Papers/Wu/mistral_all_results.json
  Summary    → /content/drive/MyDrive/Papers/Wu/mistral_summary.json
  Env info   → /content/drive/MyDrive/Papers/Wu/mistral_environment.json

FINAL RESULTS — MISTRAL 7B INSTRUCT v0.3
Config     Acc%  Easy%   Med%  Hard%  Δ(H-E)  Lat(s)   Tok/s  Mem(MB)
------------------------------------------------------------------------
BF16      32.7%  32.0%  46.0%  20.0%  -12.0%   13.61    16.6    18735
INT8      39.3%  34.0%  56.0%  28.0%   -6.0%   41.22     5.5    12263
NF4       30.7%  34.0%  40.0%  18.0%  -16.0%   15.33    14.8     8484
GPTQ      34.0%  40.0%  42.0%  20.0%  -20.0%   27.51     8.2     8521

✅ Done! All fi